# Voxtral TTS + Shannon-Prime VHT2 KV Cache Compression

This notebook benchmarks **Voxtral-4B TTS** (Mistral's text-to-speech model) with and without
**Shannon-Prime VHT2 KV cache compression**.

Architecture: 26-layer Mistral backbone, GQA (32 query heads, 8 KV heads), head_dim=128.

Ship defaults: K bands 5/5/4/3, V flat 3 -> ~4.6x KV cache compression.

**Requirements:** GPU runtime (T4 minimum, A100/L4 preferred).

## 1. Install Dependencies

In [ ]:
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q mistral_common safetensors huggingface_hub soundfile matplotlib numpy

## 2. Clone the Forked Repo

In [ ]:
import os

REPO_DIR = "/content/ComfyUI-FL-VoxtralTTS"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/nihilistau/ComfyUI-FL-VoxtralTTS.git {REPO_DIR}
else:
    print(f"Repo already cloned at {REPO_DIR}")

# Add src/ to path so voxtral_tts package is importable
import sys
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
sys.path.insert(0, REPO_DIR)

## 3. Download Model

Downloads `mistralai/Voxtral-4B-TTS-2603` via huggingface_hub. ~8 GB.

In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download

MODEL_REPO_ID = "mistralai/Voxtral-4B-TTS-2603"
MODEL_DIR = Path("/content/models/VoxtralTTS")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

marker = MODEL_DIR / "consolidated.safetensors"
if not marker.exists():
    print(f"Downloading {MODEL_REPO_ID}...")
    snapshot_download(
        repo_id=MODEL_REPO_ID,
        local_dir=str(MODEL_DIR),
        local_dir_use_symlinks=False,
    )
    print("Download complete.")
else:
    print(f"Model already present at {MODEL_DIR}")

## 4. Build Pipeline (Standalone, No ComfyUI)

We load the pipeline directly from the repo's source modules, bypassing ComfyUI.

In [ ]:
import torch
import numpy as np
from safetensors.torch import load_file

from voxtral_tts.config import VoxtralConfig
from voxtral_tts.backbone import MistralBackbone
from voxtral_tts.acoustic_transformer import FlowMatchingAcousticTransformer
from voxtral_tts.codec_decoder import VoxtralCodecDecoder
from voxtral_tts.embeddings import MultiVocabEmbeddings
from voxtral_tts.tokenizer import VoxtralTokenizer
from voxtral_tts.pipeline import VoxtralTTSPipeline

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.bfloat16
print(f"Device: {DEVICE}, dtype: {DTYPE}")


def _remap_weights(state_dict: dict) -> dict:
    """Remap Mistral weight names to module structure."""
    parts = {
        "backbone": {},
        "acoustic_transformer": {},
        "codec_decoder": {},
        "audio_embeddings": {},
    }
    for key, value in state_dict.items():
        if key.startswith("acoustic_transformer."):
            parts["acoustic_transformer"][key[len("acoustic_transformer."):]] = value
        elif key.startswith("audio_tokenizer."):
            parts["codec_decoder"][key[len("audio_tokenizer."):]] = value
        elif key.startswith("mm_audio_embeddings.audio_codebook_embeddings."):
            parts["audio_embeddings"][key[len("mm_audio_embeddings.audio_codebook_embeddings."):]] = value
        elif key.startswith("mm_audio_embeddings.tok_embeddings."):
            parts["backbone"][key[len("mm_audio_embeddings."):]] = value
        else:
            parts["backbone"][key] = value
    return parts


def load_pipeline() -> VoxtralTTSPipeline:
    """Load the full Voxtral TTS pipeline from disk."""
    config = VoxtralConfig.from_params_json(MODEL_DIR / "params.json")

    print("Loading weights...")
    raw_weights = load_file(str(MODEL_DIR / "consolidated.safetensors"))
    parts = _remap_weights(raw_weights)
    del raw_weights

    print("Building backbone...")
    backbone = MistralBackbone(config.transformer)
    backbone.load_state_dict(parts["backbone"], strict=False)
    backbone.to(device=DEVICE, dtype=DTYPE)
    backbone.eval()

    print("Building acoustic transformer...")
    acoustic_tf = FlowMatchingAcousticTransformer(config.acoustic_transformer)
    acoustic_tf.load_state_dict(parts["acoustic_transformer"], strict=False)
    acoustic_tf.to(device=DEVICE, dtype=DTYPE)
    acoustic_tf.eval()

    print("Building codec decoder...")
    codec = VoxtralCodecDecoder(config.codec_decoder)
    codec.load_state_dict(parts["codec_decoder"], strict=False)
    codec.to(device=DEVICE, dtype=DTYPE)
    codec.eval()

    print("Building audio embeddings...")
    emb_weight = parts["audio_embeddings"].get("embeddings.weight")
    total_entries = emb_weight.shape[0] if emb_weight is not None else 9088
    audio_emb = MultiVocabEmbeddings.from_config(
        total_entries=total_entries, embedding_dim=config.transformer.dim
    )
    audio_emb.load_state_dict(parts["audio_embeddings"], strict=False)
    audio_emb.to(device=DEVICE, dtype=DTYPE)
    audio_emb.eval()
    del parts

    print("Loading tokenizer...")
    tokenizer = VoxtralTokenizer.from_model_dir(MODEL_DIR)

    pipeline = VoxtralTTSPipeline(
        backbone=backbone,
        acoustic_transformer=acoustic_tf,
        codec_decoder=codec,
        audio_embeddings=audio_emb,
        tokenizer=tokenizer,
        config=config,
        voice_embeddings_dir=MODEL_DIR / "voice_embedding",
        device=DEVICE,
        dtype=DTYPE,
    )
    print("Pipeline ready.")
    return pipeline


pipeline = load_pipeline()

## 5. Benchmark Utilities

In [ ]:
import time
import soundfile as sf

TEST_TEXT = (
    "Shannon-Prime compresses the KV cache using spectral-domain banded "
    "quantization. The Vilenkin-Hartley transform factorizes perfectly "
    "at head dimension one hundred twenty-eight, yielding four point six "
    "times compression with zero quality loss on generated speech."
)
TEST_VOICE = "casual_male"
TEST_SEED = 42
MAX_FRAMES = 2048


def measure_generation(pipeline, label="baseline", save_path=None):
    """Run generation and measure GPU memory + wall time.

    Returns dict with peak_mem_mb, elapsed_s, audio_np, sample_rate.
    """
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    mem_before = torch.cuda.memory_allocated() / 1e6

    t0 = time.perf_counter()
    audio_np, sr = pipeline.generate(
        text=TEST_TEXT,
        voice=TEST_VOICE,
        max_frames=MAX_FRAMES,
        seed=TEST_SEED,
        cfg_alpha=1.2,
        noise_scale=1.0,
        euler_steps=8,
    )
    elapsed = time.perf_counter() - t0

    peak_mem = torch.cuda.max_memory_allocated() / 1e6

    if save_path:
        sf.write(save_path, audio_np, sr)
        print(f"  [{label}] Saved audio to {save_path}")

    duration_s = len(audio_np) / sr
    print(f"  [{label}] Peak GPU: {peak_mem:.1f} MB | Time: {elapsed:.2f}s | "
          f"Audio: {duration_s:.2f}s ({len(audio_np)} samples @ {sr} Hz)")

    return {
        "label": label,
        "peak_mem_mb": peak_mem,
        "mem_before_mb": mem_before,
        "elapsed_s": elapsed,
        "audio_np": audio_np,
        "sample_rate": sr,
        "duration_s": duration_s,
    }

## 6. Run Baseline (No Shannon-Prime)

In [ ]:
print("=" * 60)
print("BASELINE: Voxtral TTS without Shannon-Prime")
print("=" * 60)

baseline = measure_generation(
    pipeline,
    label="baseline",
    save_path="/content/voxtral_baseline.wav",
)

## 7. Apply Shannon-Prime VHT2 KV Compression

Monkey-patches the backbone to route KV through VHT2 compress/decompress.

In [ ]:
# Import Shannon-Prime KV compression from the forked repo
sys.path.insert(0, REPO_DIR)
from nodes.shannon_prime_kv import (
    ShannonPrimeKVCache,
    patch_voxtral_backbone,
)


def apply_sp_compression(pipeline, k_bits_str="5,5,4,3", v_bits_str="3"):
    """Apply Shannon-Prime VHT2 KV compression to pipeline backbone.

    Returns the ShannonPrimeKVCache instance for flush/stats.
    """
    k_bits = [int(x.strip()) for x in k_bits_str.split(",")]
    v_bits = [int(x.strip()) for x in v_bits_str.split(",")]

    config = pipeline.config
    sp_cache = ShannonPrimeKVCache(
        n_layers=config.transformer.n_layers,   # 26
        head_dim=config.transformer.head_dim,    # 128
        k_bands=len(k_bits),
        k_bits=k_bits,
        v_bands=len(v_bits),
        v_bits=v_bits,
        enabled=True,
    )

    patch_voxtral_backbone(pipeline.backbone, sp_cache)
    print(f"Shannon-Prime VHT2 applied: K={k_bits}, V={v_bits}")
    return sp_cache


def remove_sp_compression(pipeline, original_forward):
    """Restore original backbone forward."""
    pipeline.backbone.forward = original_forward
    print("Shannon-Prime compression removed (original forward restored).")


# Save original forward for restoring later
original_forward = pipeline.backbone.forward

In [ ]:
print("=" * 60)
print("SHANNON-PRIME: VHT2 KV compression (ship defaults 5/5/4/3 + V=3)")
print("=" * 60)

sp_cache = apply_sp_compression(pipeline, k_bits_str="5,5,4,3", v_bits_str="3")

sp_ship = measure_generation(
    pipeline,
    label="SP-5/5/4/3",
    save_path="/content/voxtral_sp_ship.wav",
)

# Flush cache for next run
sp_cache.reset()

## 8. Compare Results

In [ ]:
import matplotlib.pyplot as plt

mem_reduction = baseline["peak_mem_mb"] - sp_ship["peak_mem_mb"]
mem_ratio = baseline["peak_mem_mb"] / sp_ship["peak_mem_mb"] if sp_ship["peak_mem_mb"] > 0 else float("inf")
speed_diff = sp_ship["elapsed_s"] - baseline["elapsed_s"]
speed_pct = (speed_diff / baseline["elapsed_s"]) * 100

print("=" * 60)
print("COMPARISON: Baseline vs Shannon-Prime (5/5/4/3)")
print("=" * 60)
print(f"Peak GPU Memory:")
print(f"  Baseline:       {baseline['peak_mem_mb']:.1f} MB")
print(f"  Shannon-Prime:  {sp_ship['peak_mem_mb']:.1f} MB")
print(f"  Reduction:      {mem_reduction:.1f} MB ({mem_ratio:.2f}x ratio)")
print(f"")
print(f"Inference Time:")
print(f"  Baseline:       {baseline['elapsed_s']:.2f}s")
print(f"  Shannon-Prime:  {sp_ship['elapsed_s']:.2f}s")
print(f"  Difference:     {speed_diff:+.2f}s ({speed_pct:+.1f}%)")
print(f"")
print(f"Audio Duration:")
print(f"  Baseline:       {baseline['duration_s']:.2f}s")
print(f"  Shannon-Prime:  {sp_ship['duration_s']:.2f}s")

In [ ]:
# Waveform overlay plot
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

sr = baseline["sample_rate"]
b_audio = baseline["audio_np"]
s_audio = sp_ship["audio_np"]

# Pad shorter one to match lengths for overlay
max_len = max(len(b_audio), len(s_audio))
b_padded = np.pad(b_audio, (0, max_len - len(b_audio)))
s_padded = np.pad(s_audio, (0, max_len - len(s_audio)))
t = np.arange(max_len) / sr

# Plot 1: Baseline waveform
axes[0].plot(t, b_padded, color="#2196F3", alpha=0.8, linewidth=0.3)
axes[0].set_title("Baseline (no compression)", fontsize=12)
axes[0].set_ylabel("Amplitude")
axes[0].set_xlim(0, t[-1])

# Plot 2: Shannon-Prime waveform
axes[1].plot(t, s_padded, color="#FF5722", alpha=0.8, linewidth=0.3)
axes[1].set_title("Shannon-Prime VHT2 (5/5/4/3 + V=3)", fontsize=12)
axes[1].set_ylabel("Amplitude")
axes[1].set_xlim(0, t[-1])

# Plot 3: Overlay
axes[2].plot(t, b_padded, color="#2196F3", alpha=0.5, linewidth=0.3, label="Baseline")
axes[2].plot(t, s_padded, color="#FF5722", alpha=0.5, linewidth=0.3, label="Shannon-Prime")
axes[2].set_title("Overlay Comparison", fontsize=12)
axes[2].set_ylabel("Amplitude")
axes[2].set_xlabel("Time (s)")
axes[2].set_xlim(0, t[-1])
axes[2].legend()

plt.tight_layout()
plt.savefig("/content/voxtral_waveform_comparison.png", dpi=150)
plt.show()
print("Saved waveform comparison to /content/voxtral_waveform_comparison.png")

## 9. Test Different Bit Allocations

Compare three configurations:
- **Ship (5/5/4/3)**: production default, ~4.6x compression
- **Aggressive (4/4/3/3)**: higher compression, slight quality tradeoff
- **Floor (3/3/3/3)**: maximum compression, quality floor

In [ ]:
BIT_CONFIGS = [
    {"name": "ship",       "k_bits": "5,5,4,3", "v_bits": "3"},
    {"name": "aggressive", "k_bits": "4,4,3,3", "v_bits": "3"},
    {"name": "floor",      "k_bits": "3,3,3,3", "v_bits": "3"},
]

results = {"baseline": baseline}

for cfg in BIT_CONFIGS:
    label = f"SP-{cfg['name']}"
    print(f"\n{'=' * 60}")
    print(f"{label}: K={cfg['k_bits']}, V={cfg['v_bits']}")
    print(f"{'=' * 60}")

    # Restore original forward before re-patching
    pipeline.backbone.forward = original_forward

    sp_cache = apply_sp_compression(pipeline, k_bits_str=cfg["k_bits"], v_bits_str=cfg["v_bits"])

    result = measure_generation(
        pipeline,
        label=label,
        save_path=f"/content/voxtral_{cfg['name']}.wav",
    )
    results[cfg["name"]] = result

    sp_cache.reset()

# Restore original forward
pipeline.backbone.forward = original_forward

In [ ]:
# Summary table
print("\n" + "=" * 75)
print(f"{'Config':<20} {'Peak GPU (MB)':>14} {'Time (s)':>10} {'Audio (s)':>10} {'Mem Ratio':>10}")
print("-" * 75)

baseline_mem = results["baseline"]["peak_mem_mb"]

for key in ["baseline", "ship", "aggressive", "floor"]:
    r = results[key]
    ratio = baseline_mem / r["peak_mem_mb"] if r["peak_mem_mb"] > 0 else 0
    name = {
        "baseline": "Baseline (no SP)",
        "ship": "SP 5/5/4/3 (ship)",
        "aggressive": "SP 4/4/3/3 (aggr)",
        "floor": "SP 3/3/3/3 (floor)",
    }[key]
    print(f"{name:<20} {r['peak_mem_mb']:>14.1f} {r['elapsed_s']:>10.2f} {r['duration_s']:>10.2f} {ratio:>9.2f}x")

print("=" * 75)

In [ ]:
# Bar chart comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

labels = ["Baseline", "Ship\n5/5/4/3", "Aggressive\n4/4/3/3", "Floor\n3/3/3/3"]
keys = ["baseline", "ship", "aggressive", "floor"]
colors = ["#2196F3", "#4CAF50", "#FF9800", "#F44336"]

mems = [results[k]["peak_mem_mb"] for k in keys]
times = [results[k]["elapsed_s"] for k in keys]

ax1.bar(labels, mems, color=colors)
ax1.set_ylabel("Peak GPU Memory (MB)")
ax1.set_title("Memory Usage")
for i, v in enumerate(mems):
    ax1.text(i, v + max(mems) * 0.01, f"{v:.0f}", ha="center", fontsize=10)

ax2.bar(labels, times, color=colors)
ax2.set_ylabel("Inference Time (s)")
ax2.set_title("Generation Speed")
for i, v in enumerate(times):
    ax2.text(i, v + max(times) * 0.01, f"{v:.1f}", ha="center", fontsize=10)

plt.suptitle("Voxtral TTS: Shannon-Prime VHT2 KV Compression Benchmark", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("/content/voxtral_benchmark_bars.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Multi-waveform comparison for all bit allocations
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

plot_data = [
    ("Baseline (no compression)", results["baseline"], "#2196F3"),
    ("Shannon-Prime 5/5/4/3 (ship)", results["ship"], "#4CAF50"),
    ("Shannon-Prime 4/4/3/3 (aggressive)", results["aggressive"], "#FF9800"),
    ("Shannon-Prime 3/3/3/3 (floor)", results["floor"], "#F44336"),
]

global_max_len = max(len(r[1]["audio_np"]) for r in plot_data)

for ax, (title, r, color) in zip(axes, plot_data):
    audio = r["audio_np"]
    padded = np.pad(audio, (0, global_max_len - len(audio)))
    t = np.arange(global_max_len) / r["sample_rate"]
    ax.plot(t, padded, color=color, alpha=0.8, linewidth=0.3)
    ax.set_title(title, fontsize=11)
    ax.set_ylabel("Amplitude")
    ax.set_xlim(0, t[-1])

axes[-1].set_xlabel("Time (s)")
plt.suptitle("Voxtral TTS: Waveform Comparison Across Bit Allocations", fontsize=14)
plt.tight_layout()
plt.savefig("/content/voxtral_waveforms_all.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Playback (Colab Audio Widget)

In [ ]:
from IPython.display import Audio, display, HTML

for key, name in [("baseline", "Baseline"), ("ship", "Ship 5/5/4/3"),
                   ("aggressive", "Aggressive 4/4/3/3"), ("floor", "Floor 3/3/3/3")]:
    r = results[key]
    display(HTML(f"<h4>{name}</h4>"))
    display(Audio(r["audio_np"], rate=r["sample_rate"]))

## 11. Save Results to Drive

In [ ]:
from google.colab import drive
import json
import shutil

drive.mount("/content/drive")

SAVE_DIR = Path("/content/drive/MyDrive/voxtral_sp_benchmark")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Save audio files
for name in ["baseline", "ship", "aggressive", "floor"]:
    r = results[name]
    sf.write(str(SAVE_DIR / f"voxtral_{name}.wav"), r["audio_np"], r["sample_rate"])

# Save plots
for fname in ["voxtral_waveform_comparison.png", "voxtral_benchmark_bars.png", "voxtral_waveforms_all.png"]:
    src = Path("/content") / fname
    if src.exists():
        shutil.copy2(str(src), str(SAVE_DIR / fname))

# Save numeric results as JSON
summary = {}
for key in ["baseline", "ship", "aggressive", "floor"]:
    r = results[key]
    summary[key] = {
        "peak_mem_mb": round(r["peak_mem_mb"], 1),
        "elapsed_s": round(r["elapsed_s"], 2),
        "duration_s": round(r["duration_s"], 2),
        "num_samples": len(r["audio_np"]),
        "sample_rate": r["sample_rate"],
    }

with open(str(SAVE_DIR / "benchmark_results.json"), "w") as f:
    json.dump(summary, f, indent=2)

print(f"All results saved to {SAVE_DIR}")
print(f"Contents:")
for p in sorted(SAVE_DIR.iterdir()):
    print(f"  {p.name} ({p.stat().st_size / 1024:.0f} KB)")

## 12. Cleanup

In [ ]:
# Restore original forward and free memory
pipeline.backbone.forward = original_forward
torch.cuda.empty_cache()
print(f"GPU memory after cleanup: {torch.cuda.memory_allocated() / 1e6:.1f} MB")